# Semantic Segmentation with DeepLabV3 on Pascal VOC
This notebook demonstrates how to finetune and evaluate a pretrained DeepLabV3 on the Pascal VOC dataset for Semantic Segmentation.

In [ ]:
import os

# Automatically detect the execution environment to configure the saving directory
if 'KAGGLE_URL_BASE' in os.environ:
    print("Kaggle Environment Detected")
    save_dir = '/kaggle/working/DL_Assignment_2'
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Colab Environment Detected")
        save_dir = '/content/drive/My Drive/DL_Assignment_2'
    except ImportError:
        print("Local Environment Detected")
        save_dir = './DL_Assignment_2'

os.makedirs(save_dir, exist_ok=True)
print(f"Results will be saved at: {save_dir}")

In [ ]:
# Install and import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.datasets import VOCSegmentation
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

## 1. Data Preparation
Load the Pascal VOC 2012 dataset and initialize the DataLoader.

In [ ]:
# Data augmentation and processing pipelines for Semantic Segmentation
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class Resize:
    def __init__(self, size):
        self.size = size
    def __call__(self, image, target):
        image = T.functional.resize(image, self.size, interpolation=T.InterpolationMode.BILINEAR)
        target = T.functional.resize(target, self.size, interpolation=T.InterpolationMode.NEAREST)
        return image, target

class ToTensor:
    def __call__(self, image, target):
        image = T.functional.to_tensor(image)
        target = torch.as_tensor(np.array(target), dtype=torch.int64)
        # Ignore the boundary class (value 255) during loss computation
        target[target == 255] = 21 
        return image, target

class Normalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, image, target):
        image = T.functional.normalize(image, mean=self.mean, std=self.std)
        return image, target

transform = Compose([
    Resize((256, 256)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Wrapper class to apply custom transformations on both Image and Mask
class VOCDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        img, target = self.dataset[idx]
        if self.transform is not None:
            img, target = self.transform(img, target)
        return img, target

# Download Dataset
train_dataset_raw = VOCSegmentation(root='./data', year='2012', image_set='train', download=True)
val_dataset_raw = VOCSegmentation(root='./data', year='2012', image_set='val', download=True)

train_dataset = VOCDatasetWrapper(train_dataset_raw, transform)
val_dataset = VOCDatasetWrapper(val_dataset_raw, transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 2. Model Initialization
Initialize the DeepLabV3 model utilizing a pre-trained ResNet50 backbone.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pre-trained DeepLabV3 model
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)

# Adjust the final classifier layer to accommodate 21 classes (20 classes + 1 background)
# VOC defines class 21 for object boundaries, which can be ignored in the loss function.
# Replace the classification head:
model.classifier[4] = nn.Conv2d(256, 22, kernel_size=(1, 1), stride=(1, 1)) # 21 class + 1 class border
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=21) # Ignore index 21 (border class mapped from 255)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 3. Training & Evaluation Pipeline
Calculate the mIoU metric and visualize Loss/mIoU learning curves.

In [ ]:
def calculate_miou(preds, labels, num_classes=21):
    preds = preds.argmax(dim=1)
    ious = []
    
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls
        
        intersection = (pred_inds[target_inds]).long().sum().item()
        union = pred_inds.long().sum().item() + target_inds.long().sum().item() - intersection
        
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(float(intersection) / float(max(union, 1)))
            
    # Exclude NaN values from Mean IoU computation
    ious = [iou for iou in ious if not np.isnan(iou)]
    return np.mean(ious) if ious else 0.0

epochs = 20
train_losses = []
val_mious = []
start_epoch = 0
checkpoint_path = os.path.join(save_dir, 'semantic_checkpoint.pth')
best_model_path = os.path.join(save_dir, 'semantic_best_deeplabv3_voc.pth')

best_miou = 0.0

import os
if os.path.exists(checkpoint_path):
    print("Found checkpoint, loading to resume training...")
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    val_mious = checkpoint['val_mious']
    
    # Restore Early stopping states
    best_miou = checkpoint.get('best_miou', 0.0)
    print(f"Resuming training from epoch {start_epoch + 1}")

for epoch in range(start_epoch, epochs):
    # Training
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]'):
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Evaluation
    model.eval()
    running_miou = 0.0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]'):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)['out']
            running_miou += calculate_miou(outputs, targets)
            
    avg_test_miou = running_miou / len(val_loader)
    val_mious.append(avg_test_miou)
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_train_loss:.4f}, mIoU: {avg_test_miou:.4f}")

    # BEST MODEL SAVING LOGIC
    if avg_test_miou > best_miou:
        best_miou = avg_test_miou
        # Save optimal model
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Optimal model saved with mIoU: {best_miou:.4f}")

    # SAVE TRAINING CHECKPOINT
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'val_mious': val_mious,
        'best_miou': best_miou
    }, checkpoint_path)

# Save the final model epoch state
torch.save(model.state_dict(), os.path.join(save_dir, 'semantic_deeplabv3_voc_last.pth'))
print("Training completed! The optimal model weights are saved at:", best_model_path)

## 4. Visualization & Analysis

In [ ]:
# Plot Training Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), train_losses, marker='o', color='blue')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs+1), val_mious, marker='o', color='orange')
plt.title('Validation mIoU')
plt.xlabel('Epoch')
plt.ylabel('mIoU')
plt.grid(True)
plt.savefig(os.path.join(save_dir, 'semantic_training_curves.png'))
plt.show()

In [ ]:
# Visualization function
def decode_segmap(image, num_classes=21):
    label_colors = np.array([
        (0,0,0), (128,0,0), (0,128,0), (128,128,0), (0,0,128),
        (128,0,128), (0,128,128), (128,128,128), (64,0,0), (192,0,0),
        (64,128,0), (192,128,0), (64,0,128), (192,0,128), (64,128,128),
        (192,128,128), (0,64,0), (128,64,0), (0,192,0), (128,192,0),
        (0,64,128), (224,224,192) # class 21 representing boundary
    ])
    r = np.zeros_like(image).astype(np.uint8)
    g = np.zeros_like(image).astype(np.uint8)
    b = np.zeros_like(image).astype(np.uint8)
    for l in range(0, num_classes + 1):
        idx = image == l
        r[idx] = label_colors[l, 0]
        g[idx] = label_colors[l, 1]
        b[idx] = label_colors[l, 2]
    rgb = np.stack([r, g, b], axis=2)
    return rgb

# Filter validation samples to identify images with multiple objects of the same class for rigorous testing
print("Executing automated search for complex overlapping object scenarios...")
target_indices = []
for i in range(len(val_dataset)):
    img, target = val_dataset[i]
    if len(torch.unique(target)) >= 3: # Có ít nhất Background + 2 loại object
        target_indices.append(i)
        if len(target_indices) >= 4:
            break

if len(target_indices) == 0:
    target_indices = [0, 1, 2, 3]

# Select filtered images
images = torch.stack([val_dataset[i][0] for i in target_indices]).to(device)
targets = np.stack([val_dataset[i][1] for i in target_indices])

model.eval()
with torch.no_grad():
    outputs = model(images)['out']
    preds = outputs.argmax(dim=1).cpu().numpy()

images = images.cpu()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axs = plt.subplots(len(images), 3, figsize=(15, 12))
for i in range(len(images)):
    img = images[i].permute(1, 2, 0).numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    axs[i, 0].imshow(img)
    if i == 0: axs[i, 0].set_title("Input Image")
    axs[i, 0].axis('off')
    
    gt_map = decode_segmap(targets[i])
    axs[i, 1].imshow(gt_map)
    if i == 0: axs[i, 1].set_title("Ground Truth Mask")
    axs[i, 1].axis('off')
    
    pred_map = decode_segmap(preds[i])
    axs[i, 2].imshow(pred_map)
    if i == 0: axs[i, 2].set_title("DeepLabV3 Predicted Mask")
    axs[i, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'semantic_predictions.png'))
plt.show()

## 5. Final Model Evaluation (Semantic Metrics)\nLoad the optimal checkpoint to compute the evaluation metrics (mIoU, Mean PA). No re-training is necessary.

In [ ]:
import numpy as np
import os
from tqdm import tqdm

if os.path.exists(best_model_path):
    print(f"Loading best weights from: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=False))
else:
    print("Warning: Optimal checkpoint not found. Evaluating using current model state.")

model.to(device)
model.eval()

# Initialize the Confusion Matrix
num_classes = 21
conf_mat = np.zeros((num_classes, num_classes))

print("Executing comprehensive evaluation on the validation dataset...")
with torch.no_grad():
    for images, targets in tqdm(val_loader, desc='Final Validation'):
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)['out']
        preds = outputs.argmax(dim=1)
        
        # # Exclude class 255 (object boundaries)
        mask = (targets >= 0) & (targets < num_classes)
        
        preds_flat = preds[mask].cpu().numpy()
        targets_flat = targets[mask].cpu().numpy()
        
        # Update the Confusion Matrix
        hist = np.bincount(num_classes * targets_flat + preds_flat, minlength=num_classes**2).reshape(num_classes, num_classes)
        conf_mat += hist

# Compute evaluation metrics
# 1. Mean Pixel Accuracy (mPA)
with np.errstate(invalid='ignore'): # Suppress divide-by-zero warnings for unrepresented classes
    class_pa = np.diag(conf_mat) / conf_mat.sum(axis=1)
mpa = np.nanmean(class_pa)

# 2. Mean Intersection over Union (mIoU)
with np.errstate(invalid='ignore'):
    iou = np.diag(conf_mat) / (conf_mat.sum(axis=1) + conf_mat.sum(axis=0) - np.diag(conf_mat))
miou = np.nanmean(iou)

print("\n" + "=" * 65)
print(f"   EVALUATION METRICS REPORT (SEMANTIC BEST MODEL)")
print("=" * 65)
print(f"[+] Mean Pixel Accuracy (mPA)    : {mpa:.4f}")
print(f"[+] Mean Intersection over Union : {miou:.4f}")
print("-" * 65)
print("--> Metrics computation finished.")


## 6. Resource Efficiency Analysis\nEvaluate the total number of parameters and calculate the inference speed (FPS).

In [ ]:
import time

# 1. Parameter Computation
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"=== COMPUTATIONAL RESOURCE REPORT (DeepLabV3) ===")
print(f"- Total Parameters: {total_params / 1e6:.2f} M")
print(f"- Trainable Parameters: {trainable_params / 1e6:.2f} M")

# 2. Inference Speed Computation (FPS)
model.eval()
print("\nEvaluating Inference Speed (FPS) across 50 sample images...")

# Hardware Warm-up execution
dummy_img = torch.randn(1, 3, 256, 256).to(device)
with torch.no_grad():
    for _ in range(5):
        model(dummy_img)

start_time = time.time()
num_test = min(50, len(val_dataset))
with torch.no_grad():
    for i in range(num_test):
        img_tensor = val_dataset[i][0].unsqueeze(0).to(device)
        _ = model(img_tensor)

if torch.cuda.is_available():
    torch.cuda.synchronize()
end_time = time.time()

total_time = end_time - start_time
fps = num_test / total_time
print(f"\n=== PERFORMANCE EVALUATION REPORT ===")
print(f"- Total inference time for {num_test} images: {total_time:.4f} giây")
print(f"- Processed Frames Per Second (FPS): {fps:.2f} FPS")
print(f"- Average latency per image: {(total_time/num_test)*1000:.2f} ms")
